# Лабораторна робота №3
## Нечітка логіка: проєктування інтелектуального модуля

**Мета роботи:** Ознайомитися з принципами нечіткої логіки та її застосуванням для проєктування інтелектуального модуля, здатного приймати рішення в умовах невизначеності.

**Завдання (Варіант 4):**
Розробити модуль для виявлення схожих записів у базі даних нерухомості Парижа. Необхідно проаналізувати колонки `description` та `adress`, щоб знайти та згрупувати об'єкти, які ймовірно є ідентичними, незважаючи на можливі орфографічні помилки, скорочення чи різні формати запису.

### Етап 1: Підготовка даних та ініціалізація метрик
На цьому етапі ми завантажуємо сирі дані, очищаємо їх від порожніх значень та ініціалізуємо функції для обчислення схожості рядків:
* **Jaro-Winkler:** для оцінки схожості текстових рядків із урахуванням перестановок.
* **Soundex:** для перевірки фонетичної схожості термінів.
* **Damerau-Levenshtein:** для визначення мінімальної кількості операцій редагування між рядками.

In [3]:
import pandas as pd
from rapidfuzz.distance import JaroWinkler
from Levenshtein import distance as damerau_levenshtein
import fuzzy

ads = pd.read_csv('./raw_data/ads.csv')
equipements = pd.read_csv('./raw_data/equipements.csv')
neighborhood = pd.read_csv('./raw_data/neighborhood_facilities.csv')
transports = pd.read_csv('./raw_data/transports.csv')

arr_of_data = [ads, equipements, neighborhood, transports]
df_lab = ads[['description', 'adress']].dropna().copy()
df_sample = df_lab.head(30).reset_index(drop=True)
print(df_sample.head())

print(f"Готовий набір даних (рядків: {len(df_sample)}):")

soundex_algo = fuzzy.Soundex(4)

def jaro_winkler_similarity(str1, str2):
    return JaroWinkler.normalized_similarity(str(str1), str(str2))

def d_levenshtein_distance(str1, str2):
    return damerau_levenshtein(str(str1), str(str2))

def soundex_similarity(str1, str2):
    try:
        code1 = soundex_algo(str(str1))
        code2 = soundex_algo(str(str2))
        return 1 if code1 == code2 else 0
    except:
        return 0

print("Функції метрик успішно ініціалізовано!")

                                         description  \
0  PARIS 17ème. AVENUE NIEL *** Vidéo disponible ...   
1  Appartement Paris 3 pièce(s) 53 m2. Stéphane P...   
2  Vente Appartement 2 pièces de 31m² - 75018 Par...   
3  A vendre, en exclusivité, dans le 11e arrondis...   
4  Gambetta 2 pièce(s) 40 m2. Rue d'Annam, à quel...   

                                              adress  
0  Achat appartement 2 pièces 63 m²Paris 17e 7501...  
1  Achat appartement 3 pièces 53 m²Paris 18e 7501...  
2  Achat appartement 2 pièces 31 m²Paris 18e 7501...  
3  Achat appartement 2 pièces 31 m²Paris 11e 7501...  
4  Achat appartement 2 pièces 40 m²Paris 20e 7502...  
Готовий набір даних (рядків: 30):
Функції метрик успішно ініціалізовано!


### Етап 2: Побудова бази правил та дефазифікація результатів

Дані успішно завантажені, а метрики готові до роботи. Тепер переходимо до основного етапу — застосування нечіткої логіки для прийняття рішень. Цей процес складається з наступних кроків:

1. **Фазифікація та агрегація:** Ми порівнюємо кожну пару записів (комбінації "кожен з кожним") за допомогою наших метрик. Оскільки для об'єктів нерухомості точність адреси є критично важливою ознакою, ми надаємо результатам порівняння адрес значно більшу вагу (50% для текстової схожості та 15% для фонетичної), ніж опису оголошення (25% та 10% відповідно). Ця логіка формує нашу базу правил нечіткого виведення.
2. **Дефазифікація:** Отриманий агрегований нечіткий результат перетворюється на чітку оцінку у відсотках (від 0 до 100%).
3. **Прийняття рішення:** Ми встановлюємо порогове значення на рівні **65%**. Якщо загальна ймовірність дорівнює або перевищує цей поріг, система класифікує записи як такі, що "Належать" до однієї групи (ідентичні об'єкти). 

Згенеруємо Топ-10 найбільш схожих записів для аналізу.

In [4]:
from tabulate import tabulate

def calculate_similarity(desc_jw, desc_soundex, addr_jw, addr_soundex):
    prob = (addr_jw * 0.50 + addr_soundex * 0.15 + 
            desc_jw * 0.25 + desc_soundex * 0.10)
    return round(prob, 2)

results = []
threshold = 65

print("Починаємо порівняння записів (це може зайняти кілька секунд)...\n")

for i in range(len(df_sample)):
    for j in range(i + 1, len(df_sample)):
        desc1, desc2 = str(df_sample.loc[i, 'description']), str(df_sample.loc[j, 'description'])
        addr1, addr2 = str(df_sample.loc[i, 'adress']), str(df_sample.loc[j, 'adress'])
        
        addr_jw = jaro_winkler_similarity(addr1, addr2)
        addr_soundex = soundex_similarity(addr1, addr2)
        addr_dl = d_levenshtein_distance(addr1, addr2)
        
        desc_jw = jaro_winkler_similarity(desc1, desc2)
        desc_soundex = soundex_similarity(desc1, desc2)
        
        probability = calculate_similarity(desc_jw, desc_soundex, addr_jw, addr_soundex)
        prob_percent = int(probability * 100)
        decision = "Належить" if prob_percent >= threshold else "Не належить"
        
        if prob_percent > 30:
            results.append([
                f"ad_{i}", f"ad_{j}",
                addr1[:25] + "...", addr2[:25] + "...",
                round(addr_jw, 2), addr_dl, 
                f"{prob_percent}%", decision
            ])

results.sort(key=lambda x: int(x[6].replace('%', '')), reverse=True)
top_10_results = results[:20]

headers = ["ID 1", "ID 2", "Адреса 1 (скороч.)", "Адреса 2 (скороч.)", "Addr JW", "Addr D-Lev", "Ймовірність", "Рішення"]
print("Топ 10 найбільш схожих записів:")
print(tabulate(top_10_results, headers=headers, tablefmt="grid", stralign="center", numalign="center"))

Починаємо порівняння записів (це може зайняти кілька секунд)...

Топ 10 найбільш схожих записів:
+--------+--------+------------------------------+------------------------------+-----------+--------------+---------------+-----------+
|  ID 1  |  ID 2  |      Адреса 1 (скороч.)      |      Адреса 2 (скороч.)      |  Addr JW  |  Addr D-Lev  |  Ймовірність  |  Рішення  |
+========+========+==============================+==============================+===========+==============+===============+===========+
|  ad_5  | ad_21  | Achat appartement 3 pièce... | Achat appartement 2 pièce... |   0.91    |      19      |      67%      | Належить  |
+--------+--------+------------------------------+------------------------------+-----------+--------------+---------------+-----------+
|  ad_0  | ad_12  | Achat appartement 2 pièce... | Achat appartement 2 pièce... |   0.91    |      17      |      66%      | Належить  |
+--------+--------+------------------------------+------------------------------+

### Висновок

Під час виконання лабораторної роботи було успішно спроєктовано та реалізовано інтелектуальний модуль на основі нечіткої логіки. Система автоматично обробила вибірку з датасету та розрахувала ступінь схожості між записами, використовуючи комбінацію фонетичних (Soundex) та текстових (Jaro-Winkler, Damerau-Levenshtein) алгоритмів порівняння.

Завдяки застосуванню продуманих вагових коефіцієнтів та фіксованого порогового значення для дефазифікації, розроблений модуль зміг коректно виявити найбільш схожі об'єкти нерухомості та відфільтрувати записи з низькою ймовірністю збігу. Це наочно демонструє головну перевагу нечіткої логіки у подібних задачах — здатність ефективно працювати з неточними чи спотвореними даними та імітувати людський процес прийняття рішень в умовах невизначеності.